<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DROID Multi-View 3D Tracking Pipeline

This notebook is a **thin orchestration layer** — all algorithm code lives in the GitHub repo.

| Stage | Script | Core Modules | Output |
|---|---|---|---|
| 1. Depth | `compute_depth.py` | `core.depth` | Stereo depth + gripper refinement |
| 2. Extrinsics | `compute_extrinsics.py` | `core.physics` | Camera-robot alignment |
| 3. Tracks | `compute_tracks.py` | `core.tracking` | Dense 3D point tracks |

**Workflow**: Clone repo → `setup.sh` → Import → Run single episode → Visualize

---
## 0. Environment Setup

In [ ]:
# @title 0a. Clone repo & install dependencies
import os

REPO_DIR = "/content/droid"
if not os.path.exists(REPO_DIR):
    !git clone --recurse-submodules https://github.com/yangyi02/droid.git {REPO_DIR}
else:
    print(f"⏭️ Repo already cloned at {REPO_DIR}")
    !cd {REPO_DIR} && git pull && git submodule update --init --recursive

%cd {REPO_DIR}
!bash setup.sh

In [ ]:
# @title 0b. Install ZED SDK (Colab only)
!apt-get update -qq
!apt-get install -y zstd
sdk_installer = "ZED_SDK_Linux_Ubuntu22.run"
!wget -q -O {sdk_installer} https://download.stereolabs.com/zedsdk/5.2/cu12/ubuntu22
!chmod +x {sdk_installer}
!./{sdk_installer} silent runtime_only skip_tools
!find /usr/local/zed/ -name "pyzed*.whl" -exec pip install {} \;
import pyzed.sl as sl
print("✅ ZED SDK installed")

In [ ]:
# @title 0c. Additional pip installs (visualization)
!pip install -q mediapy plotly polyscope pyrender PyOpenGL-accelerate open3d yourdfpy

In [ ]:
# @title 0d. Python imports & sys.path setup
import sys, os, json, random, warnings
import numpy as np
import torch
import cv2
import mediapy as media
from tqdm import tqdm

REPO_DIR = "/content/droid"
for p in [
    REPO_DIR,
    os.path.join(REPO_DIR, "third_party/s2m2/src"),
    os.path.join(REPO_DIR, "third_party/vggt"),
    os.path.join(REPO_DIR, "third_party/co-tracker"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(REPO_DIR)
os.environ['PYOPENGL_PLATFORM'] = 'egl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"✅ Device: {device}")

In [ ]:
# @title 🔄 Dev: Sync from GitHub + Hot Reload (run after pushing changes from Cider)
# This cell pulls your latest code changes WITHOUT restarting the runtime.
# All loaded models (S2M2, VGGT, CoTracker, SAM) and data stay in GPU memory.

import importlib, subprocess

# 1. Git pull
result = subprocess.run(
    ["git", "pull", "--ff-only"],
    cwd=REPO_DIR, capture_output=True, text=True)
print(result.stdout.strip() or result.stderr.strip())

# 2. Reload all core modules (order matters: geometry first, then higher-level)
import core.geometry, core.io, core.depth, core.physics, core.tracking
for mod in [core.geometry, core.io, core.depth, core.physics, core.tracking]:
    importlib.reload(mod)
print("✅ Core modules reloaded")

# 3. Reload compute scripts (they import from core, so reload after core)
import compute_depth, compute_extrinsics, compute_tracks
for mod in [compute_depth, compute_extrinsics, compute_tracks]:
    importlib.reload(mod)
print("✅ Compute scripts reloaded")

# 4. Reload visualization
import utils.visualization
importlib.reload(utils.visualization)
print("✅ Visualization reloaded")
print("\n🎯 Ready! Re-run the cells below to test your changes.")

---
## 1. Stage 1: Depth Extraction

Calls functions from `compute_depth.py` → uses `core.depth` for stereo inference + gripper refinement.

In [ ]:
# @title 1a. Initialize all foundation models (S2M2, SAM)
from compute_depth import init_all_models, load_metadata

s2m2_model, run_stereo_matching, sam_predictor = init_all_models()
id_to_path, serials_db, keep_ranges = load_metadata()

valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()))
print(f"📋 Total episodes available: {len(valid_ids)}")

In [ ]:
# @title 1b. Select & process an episode
from compute_depth import (
    init_episode, extract_svo_video,
    parse_robot_kinematics, align_temporal_streams,
)

# Pick a random episode (or set manually)
episode_id = random.choice(valid_ids)
# episode_id = "your_specific_episode_id_here"  # uncomment to override
print(f"🎯 Selected episode: {episode_id}")

scene_constants = init_episode(
    episode_id,
    os.path.expanduser("~/droid_data/input/robotics/droid_raw/1.0.1"),
    id_to_path, serials_db, keep_ranges)
scene_constants = extract_svo_video(scene_constants)
scene_constants = parse_robot_kinematics(scene_constants)
scene_constants = align_temporal_streams(scene_constants)

print(f"✅ Episode loaded: {len(list(scene_constants['camera'].keys()))} cameras")

In [ ]:
# @title 1c. Run stereo depth + gripper refinement
from core.depth import (
    compute_stereo_depth,
    build_universal_gripper_mask,
    distill_empirical_gripper_depth,
    inject_gripper_depth,
)

# S2M2 stereo depth
scene_constants = compute_stereo_depth(
    scene_constants, s2m2_model, run_stereo_matching, device)

# SAM gripper mask + depth distillation
wrist_serial = scene_constants["meta"].get("wrist_serial")
if wrist_serial and wrist_serial in scene_constants["camera"]:
    wrist_data = scene_constants["camera"][wrist_serial]
    if "raw_depth" in wrist_data:
        wrist_data["original_raw_depth"] = wrist_data["raw_depth"].copy()

scene_constants = build_universal_gripper_mask(scene_constants, sam_predictor)
scene_constants = distill_empirical_gripper_depth(scene_constants)
scene_constants = inject_gripper_depth(scene_constants)

print("✅ Stage 1 complete: depth extracted + gripper refined")

In [ ]:
# @title 1d. Export depth to disk
from compute_depth import export_to_disk

export_to_disk(scene_constants)
print("✅ Depth data saved to disk")

In [ ]:
# @title 1e. Visualize depth results
from utils.visualization import inspect_dict_structure

inspect_dict_structure(scene_constants)

for cam_id, cam_data in scene_constants["camera"].items():
    if "raw_depth" in cam_data:
        depth_video = cam_data["raw_depth"]
        vis_frames = []
        for d in depth_video:
            d_vis = np.clip(d, 0, 2.0) / 2.0 * 255
            vis_frames.append(d_vis.astype(np.uint8))
        print(f"📷 [{cam_id}] Depth shape: {depth_video.shape}")
        media.show_video(vis_frames[:30], fps=10, title=f"Depth [{cam_id}]")

---
## 2. Stage 2: Camera Extrinsics Calibration

Calls functions from `compute_extrinsics.py` → uses `core.physics.TensorRobotRenderer`.

In [ ]:
# @title 2a. Initialize calibration models (VGGT + TensorRobotRenderer)
from compute_extrinsics import init_calibration_models, load_metadata as load_meta_ext

vggt_model, load_fn, pose_fn, tensor_renderer = init_calibration_models()
_, _, _, extrinsics_db = load_meta_ext()
print("✅ Calibration models loaded")

In [ ]:
# @title 2b. Run extrinsics calibration pipeline
from core.io import load_depth_data
from compute_extrinsics import (
    init_camera_states,
    vggt_warmup_extrinsics,
    run_stage2_alignment,
    run_global_joint_alignment,
    export_extrinsics,
)

# Load previously exported depth
scene_constants = load_depth_data(scene_constants, episode_id)

# Init from dataset extrinsics
scene_state = init_camera_states(scene_constants, extrinsics_db)

# VGGT visual anchoring
scene_state = vggt_warmup_extrinsics(
    scene_constants, vggt_model, load_fn, pose_fn, device)

# Robot-camera alignment
scene_state = run_stage2_alignment(
    scene_constants, tensor_renderer, scene_state)

# Global joint optimization
scene_state = run_global_joint_alignment(
    scene_constants, scene_state, tensor_renderer)

print("✅ Stage 2 complete: extrinsics calibrated")

In [ ]:
# @title 2c. Export extrinsics
export_extrinsics(scene_constants, scene_state)
print("✅ Extrinsics saved to disk")

In [ ]:
# @title 2d. Visualize: camera axes overlay
from utils.visualization import render_cross_camera_axes

axes_frames = render_cross_camera_axes(scene_constants, scene_state)
if axes_frames:
    media.show_video(axes_frames[:30], fps=10, title="Camera Axes")

---
## 3. Stage 3: Dense 3D Point Tracking

Calls functions from `compute_tracks.py` → uses `core.tracking.URDFKinematicsTracker` + `core.physics.PyBulletRenderer`.

In [ ]:
# @title 3a. Initialize tracking models (CoTracker + PyBullet)
from compute_tracks import init_tracking_models

cotracker_model, pb_renderer = init_tracking_models()
print("✅ Tracking models loaded")

In [ ]:
# @title 3b. Run full tracking pipeline (5 phases)
from core.io import load_extrinsics
from core.tracking import URDFKinematicsTracker
from compute_tracks import (
    phase1_extract_2d_tracks,
    phase2_lift_and_filter,
    phase3_3d_dedup,
    phase4_cross_view_completion,
    phase5_median_3d_fusion,
    export_tracks,
)

# Load previously exported extrinsics
scene_state = load_extrinsics(scene_constants, episode_id)
camera_ids = list(scene_constants["camera"].keys())

# Phase 1: Per-view 2D tracking
scene_constants = phase1_extract_2d_tracks(
    cotracker_model, scene_constants, device)

# Phase 2: Lift to 3D + robot/env split
per_cam_env = phase2_lift_and_filter(
    scene_constants, scene_state, pb_renderer)

# Phase 3: Cross-view dedup
unified_pts_3d, unified_to_cam, N_unified = phase3_3d_dedup(
    per_cam_env, camera_ids)

# Phase 4: Cross-view completion
per_cam_tracks, per_cam_vis = phase4_cross_view_completion(
    cotracker_model, scene_constants, scene_state,
    per_cam_env, unified_pts_3d, unified_to_cam, N_unified, device)

# Phase 5: Median 3D fusion
(final_traj_3d, final_vis_global,
 final_per_cam_tracks, final_per_cam_vis) = phase5_median_3d_fusion(
    scene_constants, scene_state,
    per_cam_tracks, per_cam_vis, N_unified)

print(f"✅ Stage 3 complete: {final_traj_3d.shape[1]} tracked 3D points")

In [ ]:
# @title 3c. Export tracks to disk
export_tracks(
    scene_constants, scene_state,
    final_traj_3d, final_vis_global,
    final_per_cam_tracks, final_per_cam_vis)
print("✅ 3D tracks saved to disk")

In [ ]:
# @title 3d. Visualize tracking results
from utils.visualization import render_all_tracks, render_cinematic_4d_orbit

# 2D overlay per camera
for cam_id in camera_ids:
    if cam_id in final_per_cam_tracks:
        track_frames = render_all_tracks(
            scene_constants, cam_id,
            final_per_cam_tracks[cam_id],
            final_per_cam_vis[cam_id],
            max_frames=30)
        if track_frames:
            media.show_video(track_frames, fps=10, title=f"Tracks [{cam_id}]")

# 4D point cloud orbit
orbit_frames = render_cinematic_4d_orbit(
    final_traj_3d, final_vis_global, n_frames=60)
if orbit_frames:
    media.show_video(orbit_frames, fps=15, title="4D Track Cloud")

---
## 4. Verify Outputs

In [ ]:
# @title 4. Quick output verification
import glob

output_root = os.path.expanduser("~/droid_data/output/mv-tap/droid")
for stage in ["depth", "extrinsics", "tracks"]:
    stage_dir = os.path.join(output_root, stage, episode_id)
    if os.path.exists(stage_dir):
        files = glob.glob(os.path.join(stage_dir, "**/*"), recursive=True)
        total_size = sum(os.path.getsize(f) for f in files if os.path.isfile(f))
        print(f"✅ {stage:12s} | {len(files):3d} files | {total_size/1e6:.1f} MB")
    else:
        print(f"❌ {stage:12s} | not found")

---
## Summary

This notebook calls functions **directly from the repo** — zero inline algorithm code:

```
droid/
├── compute_depth.py          → Stage 1 functions
├── compute_extrinsics.py     → Stage 2 functions  
├── compute_tracks.py         → Stage 3 functions
├── core/                     → Shared modules
│   ├── geometry.py           (3D math primitives)
│   ├── io.py                 (data loading)
│   ├── depth.py              (S2M2 + SAM gripper)
│   ├── physics.py            (robot renderers)
│   └── tracking.py           (URDF kinematics tracker)
└── utils/visualization.py    → All visualization helpers
```

For batch processing on GCP, use `run_parallel.sh`.